# TV-04 — DiscoLoop : récurrence à deux canaux pour le raisonnement two-hop internalisé

**Grain #20516** (arc B #17528 · qualification #17540) — d'après *DiscoLoop* :
Fu, Guo, Wang, Zhu, Lee, Jiao, Russell, Mei (UC Berkeley, arXiv 2607.00341,
2026). PDF archivé au gisement partagé (`MachineLearning/2026 - Fu et al - DiscoLoop
- ... (arXiv 2607.00341).pdf`). Aucune figure ni texte de l'article n'est recopié ici :
ce carnet **rejoue en jouet CPU le régime** du papier, chaque renvoi est une paraphrase.

Ce que le papier établit, et que ce carnet mesure sur tâche symbolique :

1. **Stockage de profondeur** — un transformer standard doit porter un fait lu en
   couche précoce jusqu'au point de la seconde récupération ; le two-hop composé en
   un seul forward pèse sur cette contrainte.
2. **Goulot représentationnel du loop simple** — l'entité-pont (le résultat
   intermédiaire du premier saut) est *décodable* depuis l'état caché sans être
   *alignée* avec son embedding ; une intervention de lecture **sans
   ré-entraînement** referme une part de l'écart.
3. **Récurrence deux-canaux** — porter simultanément un canal **continu** (l'état
   caché) et un canal **discret** (un token ré-embeddé à chaque itération) maintient
   l'état aligné : exactitude quasi-parfaite en moins de pas d'entraînement.

Fil de la série : TV-00a (RoPE) → TV-00b (variantes d'attention) → TV-01/02 (SOTA)
→ TV-03 (internalisation du CoT, Huang et al.) → **TV-04 (boucles et canaux mixtes)**.

Prérequis : `tv/` (TV-00b/TV-03) — kernel `python3`, CPU, PyTorch seul.

## 1. La tâche : two-hop à entité-pont

L'organe `tv/task.py` (v4) porte déjà la tâche exacte du papier sous forme
symbolique : l'**indirection double**. Une séquence contient Q marqueurs ; le
marqueur en position 0 porte une **valeur-pointeur v** ; le marqueur en position
v porte une seconde valeur-pointeur w ; la cible est la valeur en position w :

- **saut 1** : lire v = valeur(M_0) — le résultat v est l'**entité-pont** ;
- **saut 2** (dépendant du premier) : lire la cible à la position v.

Composer les deux lectures en **un seul forward** est précisément le régime
two-hop internalisé de Fu et al. — la chaîne passerait par le pont, le modèle
answer-only doit le *porter* en interne. Aucune tâche n'est réimplémentée ici.

In [1]:
import sys, time, math, statistics
from pathlib import Path

import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd()))

from tv.task import Vocab, lot_indirection
from tv.model import PetitLM
from tv.loop import (BoucleSimple, DiscoLoop, diagnostic_alignement,
                     realignement_lecture, selfcheck_boucle_invariants)

VOCAB_TACHE = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=4)
V = VOCAB_TACHE.VOCAB + 1          # +1 : jeton QUESTION_IND (v4)
T = 24                              # longueur de séquence

gen_ex = torch.Generator().manual_seed(3)
lot_ex = lot_indirection(4, T, gen_ex, VOCAB_TACHE, double=True)
for i in range(2):
    seq = lot_ex.x[i].tolist()
    v_pont = lot_ex.q[i].item()
    print(f"exemple {i} : M_0={seq[0]} (pont v={v_pont}) | cible y={lot_ex.y[i].item()}")
print(f"vocabulaire effectif V={V}, hasard exactitude = 1/8 = {1/8:.3f}")

exemple 0 : M_0=1 (pont v=1) | cible y=1
exemple 1 : M_0=2 (pont v=2) | cible y=5
vocabulaire effectif V=24, hasard exactitude = 1/8 = 0.125


La valeur `v` lue au premier saut désigne la **position** du second saut :
chaque exemple tire son propre pont — le modèle ne peut pas mémoriser un chemin,
il doit composer les deux lectures. Le hasard est 1/8 (la cible est un marqueur
parmi 8 valeurs).

## 2. Les trois architectures (organe `tv/loop.py`)

- **PetitLM (n=3)** — transformer standard de profondeur 3 : le témoin non
  bouclé (mesure 1 du papier).
- **BoucleSimple (T=3)** — **un seul** bloc appliqué 3 fois, poids partagés :
  la profondeur effective croît sans qu'aucun paramètre ne s'ajoute.
- **DiscoLoop (T=3)** — la récurrence deux-canaux : à chaque itération le canal
  discret (token dur par position, estimateur *straight-through*) est
  ré-embeddé et fusionné dans l'état continu (`h ← Bloc(h + λ·emb(z))`), puis
  réécrit depuis la tête. Le paramètre `lam` règle l'intensité du canal discret.

Même largeur (d=64, 4 têtes), même budget de profondeur effective (3) : la
comparaison porte sur la *forme* de la récurrence, pas sur la taille.

In [2]:
def fab_petitlm():
    return PetitLM(V, 64, 4, 4, None, n_couches=3)

def fab_boucle():
    return BoucleSimple(V, 64, 4, 4, None, T=3)

def fab_disco():
    return DiscoLoop(V, 64, 4, 4, None, T=3, lam=1.0)

FABS = {"PetitLM n=3": fab_petitlm,
        "BoucleSimple T=3": fab_boucle,
        "DiscoLoop T=3": fab_disco}

for nom, fab in FABS.items():
    n = sum(p.numel() for p in fab().parameters())
    print(f"{nom:20s} {n:>7,d} parametres")

PetitLM n=3          152,384 parametres
BoucleSimple T=3      52,928 parametres
DiscoLoop T=3         52,928 parametres


La boucle T=3 compte exactement comme une couche unique : les poids sont
**partagés** entre itérations, seul `BoucleSimple`/`DiscoLoop` réutilise le
même bloc — là où `PetitLM` paie chaque couche. C'est le levier d'efficacité
que le papier revendique (moins de paramètres, moins de pas).

In [3]:
res_check = selfcheck_boucle_invariants(vocab=V, d_model=64, n_heads=4)
print("selfcheck :", "PASSE" if res_check["passed"] else "ECHEC")
for s in res_check["samples"]:
    print(" -", s["invariant"], ":", {k: v for k, v in s.items() if k not in ("invariant", "ok")},
          "OK" if s["ok"] else "KO")

selfcheck : PASSE
 - T1_equiv_petitlm : {'max_abs_diff': 0.0} OK
 - lam0_equiv_boucle : {'max_abs_diff': 0.0} OK
 - poids_partages : {'n_T1': 52928, 'n_T4': 52928, 'n_prof4': 202112} OK
 - canal_discret_dur : {'ids_valides': True, 'grad_vivant': True} OK


Quatre invariants, quatre garanties sur l'organe avant toute mesure :
(i) `BoucleSimple(T=1)` est **exactement** un `PetitLM` à une couche (mêmes
poids, mêmes logits) ; (ii) `lam=0` éteint le canal discret sans résidu ;
(iii) le compte de paramètres ne dépend pas de T ; (iv) le canal discret est
dur en forward (ids valides) et vivant en backward (le gradient atteint
l'embedding). La mesure qui suit ne peut pas être un artefact de câblage.

## 3. La mesure : 3 bras × 4 seeds (0, 1, 7, 42)

Chaque bras est entraîné from-scratch sur l'indirection double (300 pas, batch
32, Adam 3e-3), une fois par seed, puis évalué sur un tirage figé de 512
exemples (seed 99). Les quatre seeds sont les seeds canoniques du protocole du
dépôt ; moyenne et écart-type sont rendus bruts — le verdict de §5 les
confronte.

In [4]:
def entrainer_bras(fab, graine, pas=300, batch=32, lr=3e-3):
    torch.manual_seed(graine)
    m = fab()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(1234 + graine)
    t0 = time.perf_counter()
    for _ in range(pas):
        lot = lot_indirection(batch, T, gen, VOCAB_TACHE, double=True)
        perte = F.cross_entropy(m(lot.x)[:, -1], lot.y)
        opt.zero_grad(); perte.backward(); opt.step()
    return m, time.perf_counter() - t0

@torch.no_grad()
def evaluer(m, n=512):
    gen_ev = torch.Generator().manual_seed(99)
    lot = lot_indirection(n, T, gen_ev, VOCAB_TACHE, double=True)
    pred = m(lot.x)[:, -1].argmax(-1)
    return (pred == lot.y).float().mean().item(), lot

SEEDS = [0, 1, 7, 42]
mesures, modeles = {}, {}
for nom, fab in FABS.items():
    accs, secs = [], []
    for graine in SEEDS:
        m, s = entrainer_bras(fab, graine)
        acc, _ = evaluer(m)
        accs.append(acc); secs.append(s)
        modeles.setdefault(nom, []).append(m)
    mesures[nom] = {"accs": accs, "moy": statistics.fmean(accs),
                    "std": statistics.pstdev(accs), "sec": statistics.fmean(secs)}
    print(f"{nom:20s} acc {mesures[nom]['moy']:.3f} +/- {mesures[nom]['std']:.3f}"
          f"  ({mesures[nom]['sec']:.1f}s/seed)   brut {[round(a,3) for a in accs]}")

PetitLM n=3          acc 0.611 +/- 0.024  (7.8s/seed)   brut [0.604, 0.648, 0.609, 0.582]


BoucleSimple T=3     acc 0.601 +/- 0.027  (6.8s/seed)   brut [0.623, 0.631, 0.564, 0.586]


DiscoLoop T=3        acc 0.553 +/- 0.017  (8.9s/seed)   brut [0.539, 0.582, 0.549, 0.541]


### Lecture du résultat

Le transformer profond compte **152 384** paramètres, les deux architectures
bouclées **52 928** (2,9× moins) — et la boucle simple égale le profond :
0.601 ± 0.027 contre 0.611 ± 0.024, écart à l'intérieur du bruit des seeds.
C'est le premier acquis du papier qui transfère tel quel à l'échelle jouet :
le partage de poids achète la profondeur effective sans payer de paramètres.
Le classement est stable sur les 4 seeds : profond ≈ boucle simple >
**DiscoLoop (λ=1,0) à 0.553 ± 0.017**, derrière de ~5 points. L'avantage
d'exactitude du deux-canaux ne se manifeste PAS dans ce régime — la
confrontation chiffrée suit en §5.

## 4. Le goulot représentationnel : décodabilité vs alignement

Le diagnostic central du papier (mesure 2) : à la position de requête, l'état
d'un loop simple peut **lire** l'entité-pont (l'argmax de la tête la rend)
sans être **aligné** avec elle (faible cosinus entre l'état et l'embedding du
pont). Les deux nombres sont indépendants — c'est leur écart qui nomme le
goulot. Ici l'entité-pont est `v`, la valeur lue au premier saut (colonne `q`
du lot).

In [5]:
diag = {}
for nom in FABS:
    m = modeles[nom][0]          # seed 0
    acc, lot_ev = evaluer(m)
    d = diagnostic_alignement(m, lot_ev.x, ids_pont=lot_ev.q)
    diag[nom] = d
    print(f"{nom:20s} decodabilite={d['decodabilite']:.3f}  alignement={d['alignement']:.3f}")

PetitLM n=3          decodabilite=0.395  alignement=-0.039


BoucleSimple T=3     decodabilite=0.412  alignement=-0.044


DiscoLoop T=3        decodabilite=0.439  alignement=0.005


### Lecture du diagnostic

La décodabilité du pont est ~0.40–0.44 partout : l'argmax de la tête rend
l'entité-pont quatre fois sur dix seulement — cohérent avec des exactitudes
de 0.55–0.61, le two-hop composé n'est pas résolu et le pont n'est pas
stocké sous une forme directement lisible. L'**alignement** discrimine les
trois bras dans la direction du papier : nul à négatif pour le profond
(−0.039) et la boucle simple (−0.044), le deux-canaux est le **seul**
légèrement positif (+0.005). Le mécanisme annoncé — le canal discret ramène
l'état vers l'espace des embeddings — est donc mesurable **présent**, mais
l'amplitude est minuscule à cette échelle : ce n'est pas lui qui décide
l'exactitude ici.

### L'intervention sans ré-entraînement

`realignement_lecture` ne touche à aucun poids : elle remplace, à la position
de lecture, la projection linéaire de la tête par une lecture en similarité
cosinus sur les embeddings **centrés**. Si le goulot est bien un défaut
d'*alignement de la lecture* — et pas une perte d'information — l'exactitude
doit remonter sans un seul pas de gradient.

In [6]:
reals = {}
for nom in FABS:
    m = modeles[nom][0]
    acc, lot_ev = evaluer(m)
    r = realignement_lecture(m, lot_ev.x, lot_ev.y)
    reals[nom] = r
    print(f"{nom:20s} avant={r['exactitude_avant']:.3f}  apres={r['exactitude_apres']:.3f}"
          f"  (delta {r['exactitude_apres']-r['exactitude_avant']:+.3f})")

PetitLM n=3          avant=0.604  apres=0.598  (delta -0.006)


BoucleSimple T=3     avant=0.623  apres=0.613  (delta -0.010)


DiscoLoop T=3        avant=0.539  apres=0.541  (delta +0.002)


### Lecture de l'intervention

Les deltas sont ≈ 0 (−0.006, −0.010, +0.002) : l'intervention **ne referme
rien** dans ce régime, contrairement au papier. La lecture honnête de cet
écart : dans le papier, le goulot apparaît sur des boucles entraînées
longuement, dont le pont est *bien décodable* mais *mal aligné* — le déficit
porte alors sur la lecture seule. Ici le pont n'est même pas bien décodable par les lectures testées
(argmax 0.41 ; lecture cosinus neutre) : l'hypothèse cohérente est un déficit
**en amont**, dans le stockage du pont — réaligner la lecture ne récupère rien,
ce que les lectures testées lisent comme une information non décodable depuis
l'état, sans pouvoir exclure qu'elle y vive sous une forme qu'elles ne savent
pas lire. Le test
garde sa valeur de témoin : l'intervention ne détériore pas non plus — c'est
une lecture alternative, pas un artefact.

## 5. Verdict

La confrontation DiscoLoop vs BoucleSimple — même budget de paramètres, même
profondeur effective — se lit sur l'écart moyen cross-seed rapporté à sa
dispersion. Le protocole du dépôt exige le verdict écrit : **BEATS** /
**NO BEATS** / **INCONCLUSIVE** — jamais « promising ».

In [7]:
d_deltas = [a - b for a, b in zip(mesures["DiscoLoop T=3"]["accs"],
                                    mesures["BoucleSimple T=3"]["accs"])]
d_moy = statistics.fmean(d_deltas)
d_std = statistics.pstdev(d_deltas)
edge = d_moy / d_std if d_std > 0 else float("inf")

def _dm(pertes_a, pertes_b):
    d = [a - b for a, b in zip(pertes_a, pertes_b)]
    n = len(d); dbar = statistics.fmean(d)
    var = sum((x - dbar) ** 2 for x in d) / n
    dm = dbar / math.sqrt(max(var, 1e-12) / n)
    p = 2 * (1 - 0.5 * (1 + math.erf(abs(dm) / math.sqrt(2))))
    return dm, p

@torch.no_grad()
def perte_par_exemple(m, n=512):
    gen_ev = torch.Generator().manual_seed(99)
    lot = lot_indirection(n, T, gen_ev, VOCAB_TACHE, double=True)
    pred = m(lot.x)[:, -1].argmax(-1)
    return [1.0 - bool(c) for c in (pred == lot.y).tolist()]

# pooling par exemple sur les 4 seeds (memes instances d'eval)
pa = [statistics.fmean(col) for col in zip(*[
    perte_par_exemple(modeles["DiscoLoop T=3"][g]) for g in range(4)])]
pb = [statistics.fmean(col) for col in zip(*[
    perte_par_exemple(modeles["BoucleSimple T=3"][g]) for g in range(4)])]
dm_stat, dm_p = _dm(pa, pb)

print(f"DiscoLoop - BoucleSimple : delta moyen {d_moy:+.4f} (std {d_std:.4f}), edge {edge:+.2f} sigma")
print(f"DM apparie (pertes 0/1, 512 exemples) : dm {dm_stat:+.3f}, p {dm_p:.4f}")
print(f"vs PetitLM n=3 : {mesures['DiscoLoop T=3']['moy']:.3f} contre {mesures['PetitLM n=3']['moy']:.3f}")

DiscoLoop - BoucleSimple : delta moyen -0.0483 (std 0.0243), edge -1.99 sigma
DM apparie (pertes 0/1, 512 exemples) : dm +4.446, p 0.0000
vs PetitLM n=3 : 0.553 contre 0.611


### Verdict écrit

**NO BEATS.** À ce régime (T=3, λ=1.0, 300 pas, d=64, 4 seeds), DiscoLoop
est mesurablement **derrière** la boucle simple : delta moyen **−0.048**
(std 0.024), edge **−1.99 σ**, DM apparié sur 512 exemples **p < 0.0001**.
L'avantage d'exactitude du papier ne se transfère pas à l'échelle jouet.

Ce que la mesure établit malgré tout, et qui n'est pas rien :

1. **Poids partagés** : la boucle égale le profond (0.601 vs 0.611) à
   2.9× moins de paramètres — l'efficacité structurelle du loop est réelle.
2. **Direction du mécanisme** : le diagnostic d'alignement ordonne les trois
   bras comme le papier le prédit (seul le deux-canaux en alignement positif).
3. **Témoin gradué vivant** : personne ne résout la tâche (0.55–0.61 loin de
   1.0) — le two-hop à pont reste non trivial, la mesure n'est pas dégénérée.

Les leviers non explorés ici sont explicites : λ non réglé (exercice 2),
budget d'entraînement court face aux courses du papier, T fixé. Un verdict
jouet NO BEATS ne réfute pas le papier — il délimite où l'effet cesse
d'apparaître, ce qui est précisément le rôle d'une ré-implementation jouet
dans cette série (même statut que le résultat négatif v4 de TV-03).

## 6. Exercices

Trois exercices, du plus guidé au plus ouvert. Les cellules s'exécutent sans
erreur même non complétées (convention C.1) — `pass` / `print` / `return None`.

**Exercice 1 — La profondeur compte-t-elle ?** Faites varier `T` de
`BoucleSimple` (1, 2, 4, 6) à budget de paramètres constant, mesurez
l'exactitude sur 2 seeds, et dites à partir de quelle profondeur effective la
boucle égale le `PetitLM` n=3. *Indice : la profondeur effective est le levier
du stockage de profondeur — comparez aussi les temps d'entraînement.

In [8]:
# Exercice 1 : profondeur effective de la boucle
# Etape 1 : boucle sur T in (1, 2, 4, 6), seeds (0, 7) -> entrainer_bras
# Etape 2 : afficher acc moyenne par T a cote de PetitLM n=3
# Etape 3 : conclure en une phrase dans un print
resultats_ex1 = None  # TODO etudiant
print("Exercice a completer : courbe acc(T) de BoucleSimple")

Exercice a completer : courbe acc(T) de BoucleSimple


**Exercice 2 — Le canal discret en intensité.** `lam` règle le poids du
canal discret dans la fusion. Mesurez l'alignement (`diagnostic_alignement`)
et l'exactitude d'un `DiscoLoop` entraîné pour `lam` dans {0.0, 0.5, 1.0,
2.0} (1 seed suffit). *Indice : `lam=0` doit reproduire `BoucleSimple` —
l'invariant 2 du selfcheck ; à très fort `lam`, le discret écrase le continu.

In [9]:
# Exercice 2 : balayage de lam
# Etape 1 : pour chaque lam, construire/entrainer un DiscoLoop (seed 0)
# Etape 2 : evaluer exactitude + diagnostic_alignement (ids_pont=lot.q)
# Etape 3 : afficher le tableau lam -> (acc, decodabilite, alignement)
resultats_ex2 = None  # TODO etudiant
print("Exercice a completer : tableau lam vs (acc, alignement)")

Exercice a completer : tableau lam vs (acc, alignement)


**Exercice 3 — Généralisation hors distribution.** Le papier insiste sur
la *généralisation* du loop. Entraînez `DiscoLoop` et `BoucleSimple` sur des
séquences de longueur T=20, puis évaluez-les sur T=32 (même vocabulaire,
séquences plus longues : le remplissage grandit). *Indice : `lot_indirection`
prend T en argument ; l'entraînement n'a jamais vu la longueur 32.

In [10]:
# Exercice 3 : generalisation en longueur
# Etape 1 : re-entrainer les deux modeles sur T=20 (seed 0, 300 pas)
# Etape 2 : evaluer sur T=32 (tirage fige seed 99)
# Etape 3 : comparer les chutes d'exactitude relatives
resultats_ex3 = None  # TODO etudiant
print("Exercice a completer : acc(T=20 -> T=32) par architecture")

Exercice a completer : acc(T=20 -> T=32) par architecture


## 7. Ce qu'il faut retenir

À l'échelle jouet (d=64, T=3, 300 pas, seeds 0/1/7/42), le régime deux-canaux
de Fu et al. ne bat pas la boucle simple : **NO BEATS**, écart −1.99 σ, DM
p < 0.0001. Trois acquis résistent à la mesure : le **partage de poids**
(2.9× moins de paramètres à exactitude égale du profond), la **direction du
diagnostic d'alignement** (le canal discret produit le seul alignement
positif — le mécanisme est présent, son amplitude ne décide pas l'exactitude
à cette échelle), et un **témoin gradué vivant** (le two-hop à pont demeure
non résolu pour tous les bras). La frontière est documentée : là où finit
l'effet jouet commence le régime du papier — entraînements longs, λ réglé,
pont décodable-mais-mal-aligné.

**Renvois** : Fu, Guo, Wang, Zhu, Lee, Jiao, Russell, Mei — *DiscoLoop*,
arXiv 2607.00341 (2026) — PDF au gisement partagé, identité vérifiée page 1.
Série : TV-00b (briques d'attention), TV-03 (internalisation du CoT, Huang et
al. 2026, arXiv 2605.28600). Organe : `tv/loop.py` (selfcheck 4/4), tâche :
`tv/task.py` v4 (indirection double).

***

Navigation : [TV-03](TV-03-Internalisation-CoT.ipynb) ← **TV-04** — fin de la
série pour l'arc B.